# 04 — Comparaison des modèles de détection de fraude

## Objectif

Ce notebook lance la première expérience comparative avec **LightGBM**, en conservant XGBoost comme référence. Il réutilise les matrices préparées dans `02_data_preparation.ipynb` : mêmes transactions, mêmes features et même découpage temporel.

- **Train :** 2010–2017 (7 203 025 transactions)
- **Validation :** 2018 (934 599 transactions)
- **Test :** 2019 (777 339 transactions), conservé à l'écart pour l'évaluation finale après les choix sur la validation
- **Entrées :** 363 features (53 numériques + 310 colonnes One-Hot)
- **Métrique principale :** PR-AUC / average precision, car la fraude est très rare

Cette première expérience utilise une configuration de départ raisonnable, pas une recherche exhaustive d'hyperparamètres. Il faudra décrire cette limite dans le mémoire.

**Important :** exécuter les cellules une par une. La matrice d'entraînement est volumineuse et le premier chargement peut prendre du temps.


## 1. Installer LightGBM (si nécessaire)

Exécute cette cellule une fois dans l'environnement Python utilisé par les notebooks. Si LightGBM est déjà installé, pip indiquera que l'exigence est satisfaite.

In [ ]:
%pip install lightgbm

In [ ]:
%pip install joblib numpy pandas scipy scikit-learn

## 2. Importer les bibliothèques

In [ ]:
import sys
import lightgbm
import numpy
import sklearn

print("Python :", sys.executable)
print("LightGBM :", lightgbm.__version__)
print("Emplacement LightGBM :", lightgbm.__file__)
print("NumPy :", numpy.__version__)
print("Scikit-learn :", sklearn.__version__)

In [ ]:
from pathlib import Path
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
import lightgbm as lgb

print("LightGBM :", lgb.__version__)
print("NumPy :", np.__version__)
print("Pandas :", pd.__version__)

## 3. Vérifier que les matrices préparées existent

Le notebook `03_model_training.ipynb` utilise `Path("../data/processed")`. On conserve ce chemin, avec quelques chemins alternatifs pour faciliter l'ouverture depuis VS Code.

Cette cellule **ne charge pas encore** les grandes matrices. Elle vérifie d'abord que les fichiers attendus sont présents.

In [ ]:
required_files = [
    "X_train_final.npz",
    "X_validation_final.npz",
    "X_test_final.npz",
    "y_train.joblib",
    "y_validation.joblib",
    "y_test.joblib",
    "scale_pos_weight.joblib",
]

candidate_dirs = [
    Path("../data/processed"),
    Path("ml/data/processed"),
    Path("data/processed"),
]

PROCESSED_DIR = None
for directory in candidate_dirs:
    if all((directory / name).is_file() for name in required_files):
        PROCESSED_DIR = directory.resolve()
        break

print("Répertoire de travail :", Path.cwd())

if PROCESSED_DIR is None:
    print("\nAucun répertoire contenant tous les fichiers requis n'a été trouvé.")
    print("Répertoires examinés :")
    for directory in candidate_dirs:
        print(" -", directory.resolve())
    print("\nFichiers manquants par répertoire :")
    for directory in candidate_dirs:
        missing = [name for name in required_files if not (directory / name).is_file()]
        print(f"\n{directory.resolve()} :")
        print(missing if missing else "aucun")
    raise FileNotFoundError(
        "Les matrices préparées sont introuvables. Ouvre ce notebook dans le dossier "
        "des notebooks du projet et vérifie que le notebook 02 a bien sauvegardé "
        "les fichiers dans ml/data/processed."
    )

print("\nRépertoire des matrices :", PROCESSED_DIR)
for name in required_files:
    file_path = PROCESSED_DIR / name
    print(f"{name:28s} | présent | {file_path.stat().st_size / (1024**2):,.2f} MB")

## 4. Charger uniquement le train et la validation

Nous n'utilisons pas encore le test 2019. Il doit rester indépendant jusqu'à ce que les décisions de comparaison et les seuils soient fixés.

In [ ]:
X_train = sparse.load_npz(PROCESSED_DIR / "X_train_final.npz")
X_validation = sparse.load_npz(PROCESSED_DIR / "X_validation_final.npz")

y_train = np.asarray(joblib.load(PROCESSED_DIR / "y_train.joblib")).ravel().astype(np.int8)
y_validation = np.asarray(joblib.load(PROCESSED_DIR / "y_validation.joblib")).ravel().astype(np.int8)
scale_pos_weight_saved = float(joblib.load(PROCESSED_DIR / "scale_pos_weight.joblib"))

print("=== Dimensions chargées ===")
print("X_train      :", X_train.shape, "| format :", type(X_train).__name__)
print("y_train      :", y_train.shape)
print("X_validation :", X_validation.shape, "| format :", type(X_validation).__name__)
print("y_validation :", y_validation.shape)

assert X_train.shape[0] == y_train.shape[0], "Train : X et y n'ont pas le même nombre de lignes."
assert X_validation.shape[0] == y_validation.shape[0], "Validation : X et y n'ont pas le même nombre de lignes."
assert X_train.shape[1] == X_validation.shape[1] == 363, "Le nombre de features n'est pas 363 dans les deux matrices."
assert set(np.unique(y_train)).issubset({0, 1}), "La cible train doit contenir uniquement 0 et 1."
assert set(np.unique(y_validation)).issubset({0, 1}), "La cible validation doit contenir uniquement 0 et 1."

print("\n=== Répartition de la cible ===")
for label, y in [("Train", y_train), ("Validation", y_validation)]:
    counts = np.bincount(y, minlength=2)
    print(f"{label:12s} | non-fraudes : {counts[0]:,} | fraudes : {counts[1]:,} | taux de fraude : {100 * y.mean():.4f} %")

computed_scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print("\nscale_pos_weight sauvegardé :", round(scale_pos_weight_saved, 2))
print("scale_pos_weight recalculé :", round(computed_scale_pos_weight, 2))
if not np.isclose(scale_pos_weight_saved, computed_scale_pos_weight, rtol=1e-3):
    warnings.warn("Le ratio sauvegardé diffère du ratio recalculé à partir du train. Nous utiliserons le ratio recalculé.")
scale_pos_weight = computed_scale_pos_weight

## 5. Contrôle mémoire avant entraînement

Les matrices sont stockées en format sparse. La cellule ci-dessous affiche leur taille sparse et, si `psutil` est installé, la mémoire vive disponible. LightGBM aura besoin de mémoire supplémentaire pour construire ses histogrammes ; la taille des matrices ne représente donc pas toute la mémoire nécessaire.

In [ ]:
def sparse_size_gb(matrix):
    """Taille mémoire des tableaux de la matrice sparse, en Go décimaux."""
    arrays = [matrix.data, matrix.indices, matrix.indptr]
    return sum(array.nbytes for array in arrays) / (1024**3)

print(f"Mémoire sparse du train      : {sparse_size_gb(X_train):.2f} Go")
print(f"Mémoire sparse de validation : {sparse_size_gb(X_validation):.2f} Go")

try:
    import psutil
    available_gb = psutil.virtual_memory().available / (1024**3)
    print(f"Mémoire vive actuellement disponible : {available_gb:.2f} Go")
    if available_gb < 8:
        warnings.warn(
            "Moins de 8 Go de RAM sont disponibles. L'entraînement complet peut échouer ou ralentir la machine. "
            "Libère de la mémoire avant de continuer. Ne réduis pas le train sans documenter ce changement."
        )
except ImportError:
    print("psutil n'est pas installé : mémoire vive disponible non mesurée.")



## 6. Référence XGBoost déjà obtenue sur la validation

Cette ligne reprend les résultats enregistrés dans `03_model_training.ipynb`. Elle sert de repère initial ; nous n'entraînons pas XGBoost une deuxième fois dans cette première version du notebook.

In [ ]:
baseline_results = pd.DataFrame([{
    "model": "XGBoost (référence existante)",
    "validation_pr_auc": 0.762797,
    "validation_roc_auc": 0.997574,
    "threshold_0_30": 0.30,
    "precision_at_0_30": 0.777213,
    "recall_at_0_30": 0.657459,
    "f1_at_0_30": 0.712338,
    "validation_predicted_positive_at_0_30": 1378,
}])
display(baseline_results)

In [ ]:
import numpy as np
import lightgbm as lgb

print("LightGBM :", lgb.__version__)
print("NumPy :", np.__version__)

rng = np.random.default_rng(42)
X = rng.random((10_000, 20), dtype=np.float32)
y = rng.integers(0, 2, size=10_000)

model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=5,
    n_jobs=2,
    verbosity=-1
)

model.fit(X, y)
print("Test réussi.")

## 7. Entraîner LightGBM — première configuration de référence

### Pourquoi cette configuration ?

- `objective="binary"` : classification fraude / non-fraude.
- `n_estimators=300` et `learning_rate=0.05` : même nombre initial d'arbres et même taux d'apprentissage que la configuration XGBoost de référence.
- `scale_pos_weight` : calculé uniquement à partir de la cible d'entraînement pour tenir compte de la rareté des fraudes.
- `subsample=0.8` avec `subsample_freq=1` : activer réellement l'échantillonnage des observations à chaque période d'échantillonnage.
- `colsample_bytree=0.8` : échantillonnage des variables par arbre.
- `metric="average_precision"` : suivre la mesure de type PR-AUC / average precision sur la validation.

Il s'agit d'une première configuration comparable, pas encore d'un réglage optimal. Nous n'activons pas l'early stopping dans cette première expérience afin de garder une procédure simple et transparente.

In [ ]:
import time
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

lgbm_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

start = time.time()

lgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_validation, y_validation)],
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=30,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=25),
    ],
)

y_validation_proba_lgbm = lgbm_model.predict_proba(
    X_validation,
    num_iteration=lgbm_model.best_iteration_
)[:, 1]

print("\n=== LightGBM avec arrêt anticipé ===")
print("Meilleure itération :", lgbm_model.best_iteration_)
print("Durée :", round((time.time() - start) / 60, 2), "minutes")
print("PR-AUC :", round(
    average_precision_score(y_validation, y_validation_proba_lgbm), 6
))
print("ROC-AUC :", round(
    roc_auc_score(y_validation, y_validation_proba_lgbm), 6
))

In [ ]:
# lgbm_model = lgb.LGBMClassifier(
#     objective="binary",
#     n_estimators=300,
#     learning_rate=0.05,
#     num_leaves=31,
#     max_depth=-1,
#     subsample=0.8,
#     subsample_freq=1,
#     colsample_bytree=0.8,
#     scale_pos_weight=scale_pos_weight,
#     random_state=42,
#     n_jobs=-1,
#     verbosity=-1,
# )

# print(lgbm_model)

In [ ]:
# start_time = time.perf_counter()

# lgbm_model.fit(
#     X_train,
#     y_train,
#     eval_set=[(X_validation, y_validation)],
#     eval_metric="average_precision",
#     callbacks=[lgb.log_evaluation(period=50)],
# )

# training_seconds = time.perf_counter() - start_time
# print(f"\nEntraînement LightGBM terminé en {training_seconds / 60:.2f} minutes.")
# print("Nombre d'arbres entraînés :", lgbm_model.n_estimators_)

In [ ]:
# import numpy as np
# from sklearn.metrics import average_precision_score, roc_auc_score

# print("=== Dimensions ===")
# print("X_train :", X_train.shape)
# print("y_train :", y_train.shape)
# print("X_validation :", X_validation.shape)
# print("y_validation :", y_validation.shape)

# print("\n=== Proportion de fraudes ===")
# print("Train :", np.mean(np.asarray(y_train)))
# print("Validation :", np.mean(np.asarray(y_validation)))

# print("\n=== Scores prédits ===")
# print("Minimum :", y_validation_proba_lgbm.min())
# print("Maximum :", y_validation_proba_lgbm.max())
# print("Médiane :", np.median(y_validation_proba_lgbm))
# print("Quantiles :", np.quantile(
#     y_validation_proba_lgbm,
#     [0.90, 0.99, 0.999, 1.0]
# ))

# print("\n=== Métriques recalculées ===")
# print("PR-AUC :", average_precision_score(
#     y_validation, y_validation_proba_lgbm
# ))
# print("ROC-AUC :", roc_auc_score(
#     y_validation, y_validation_proba_lgbm
# ))

In [ ]:
# import numpy as np
# from sklearn.metrics import average_precision_score, roc_auc_score

# # 1. Recalculer les prédictions du modèle actuel
# scores_frais = lgbm_model.predict_proba(X_validation)[:, 1]
# scores_enregistres = np.asarray(y_validation_proba_lgbm)
# y_true = np.asarray(y_validation)

# print("=== Vérification des prédictions ===")
# print("Même longueur :", len(scores_frais) == len(scores_enregistres))
# print(
#     "Prédictions identiques :",
#     np.allclose(scores_frais, scores_enregistres)
# )
# print(
#     "Écart maximal entre les deux séries :",
#     np.max(np.abs(scores_frais - scores_enregistres))
# )

# # 2. Recalculer les métriques sur les prédictions fraîches
# print("\n=== Métriques recalculées ===")
# print(
#     "Average Precision :",
#     average_precision_score(y_true, scores_frais)
# )
# print(
#     "ROC-AUC :",
#     roc_auc_score(y_true, scores_frais)
# )

# # 3. Examiner les métriques de chaque étape d'entraînement
# print("\n=== Historique de l'entraînement ===")
# historique = getattr(lgbm_model, "evals_result_", {})

# for ensemble, metriques in historique.items():
#     print(f"\nEnsemble : {ensemble}")
#     for nom, valeurs in metriques.items():
#         print(f"{nom} :")
#         print("  50 arbres  :", valeurs[49] if len(valeurs) >= 50 else "N/A")
#         print("  100 arbres :", valeurs[99] if len(valeurs) >= 100 else "N/A")
#         print("  150 arbres :", valeurs[149] if len(valeurs) >= 150 else "N/A")
#         print("  Dernière valeur :", valeurs[-1])

# print("\n=== Informations du modèle ===")
# print("Arbres configurés :", lgbm_model.n_estimators)
# print("Arbres entraînés :", lgbm_model.booster_.num_trees())
# print("Meilleure itération :", getattr(lgbm_model, "best_iteration_", None))

## 8. Évaluer LightGBM sur la validation

On mesure d'abord la qualité du classement à partir des scores, indépendamment d'un seuil métier fixe.

In [ ]:
y_validation_proba_lgbm = lgbm_model.predict_proba(X_validation)[:, 1]

lgbm_validation_pr_auc = average_precision_score(y_validation, y_validation_proba_lgbm)
lgbm_validation_roc_auc = roc_auc_score(y_validation, y_validation_proba_lgbm)

print("=== Validation 2018 — LightGBM ===")
print(f"PR-AUC  : {lgbm_validation_pr_auc:.6f}")
print(f"ROC-AUC : {lgbm_validation_roc_auc:.6f}")
print(f"Durée entraînement : {training_seconds / 60:.2f} minutes")

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

# Vérifier les probabilités LightGBM déjà calculées
print(
    "PR-AUC des probabilités disponibles :",
    round(average_precision_score(
        y_validation, y_validation_proba_lgbm
    ), 6)
)

# Calculer les métriques au seuil 0,30, sans réentraîner le modèle
threshold_lgbm = 0.30
y_validation_pred_lgbm = (
    y_validation_proba_lgbm >= threshold_lgbm
).astype(int)

precision_lgbm = precision_score(
    y_validation, y_validation_pred_lgbm, zero_division=0
)
recall_lgbm = recall_score(
    y_validation, y_validation_pred_lgbm, zero_division=0
)
f1_lgbm = f1_score(
    y_validation, y_validation_pred_lgbm, zero_division=0
)

print("\n=== LightGBM — Validation 2018, seuil 0,30 ===")
print(f"Précision : {precision_lgbm:.4%}")
print(f"Rappel : {recall_lgbm:.4%}")
print(f"F1 : {f1_lgbm:.6f}")
print(f"Transactions signalées : {y_validation_pred_lgbm.sum():,}")

## 9. Comparer plusieurs seuils sur la validation

Pour une comparaison métier, nous mesurons la précision, le rappel, le F1-score, les faux positifs et les fraudes manquées. Le seuil optimal selon le F1 sera identifié sur la validation uniquement. Le seuil ne sera pas automatiquement identique à celui de XGBoost.

In [ ]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
threshold_rows = []

for threshold in thresholds:
    y_pred = (y_validation_proba_lgbm >= threshold).astype(np.int8)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_validation, y_pred, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_validation, y_pred, labels=[0, 1]).ravel()
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "predicted_for_otp": int(y_pred.sum()),
        "false_positives": int(fp),
        "frauds_missed": int(fn),
        "frauds_detected": int(tp),
    })

threshold_results_lgbm = pd.DataFrame(threshold_rows)
display(threshold_results_lgbm.style.format({
    "threshold": "{:.2f}",
    "precision": "{:.4%}",
    "recall": "{:.4%}",
    "f1_score": "{:.4f}",
}))

best_row = threshold_results_lgbm.loc[threshold_results_lgbm["f1_score"].idxmax()]
best_threshold_lgbm = float(best_row["threshold"])
print(f"Seuil au F1 maximal sur validation : {best_threshold_lgbm:.2f}")
print("À noter : ce seuil est un choix selon le F1, pas encore un seuil métier validé par les coûts réels.")

## 10. Comparaison initiale XGBoost / LightGBM sur validation

La comparaison utilise la PR-AUC et la ROC-AUC pour la qualité de classement. Les métriques au seuil 0,30 servent à comparer les deux modèles au même seuil ; le meilleur seuil F1 de LightGBM est présenté séparément.

In [ ]:
row_at_030 = threshold_results_lgbm.loc[np.isclose(threshold_results_lgbm["threshold"], 0.30)].iloc[0]

lgbm_result = {
    "model": "LightGBM (première configuration)",
    "validation_pr_auc": lgbm_validation_pr_auc,
    "validation_roc_auc": lgbm_validation_roc_auc,
    "threshold_0_30": 0.30,
    "precision_at_0_30": float(row_at_030["precision"]),
    "recall_at_0_30": float(row_at_030["recall"]),
    "f1_at_0_30": float(row_at_030["f1_score"]),
    "validation_predicted_positive_at_0_30": int(row_at_030["predicted_for_otp"]),
    "best_f1_threshold_validation": best_threshold_lgbm,
    "best_f1_validation": float(best_row["f1_score"]),
    "training_minutes": training_seconds / 60,
}

comparison_results = pd.concat([
    baseline_results,
    pd.DataFrame([lgbm_result]),
], ignore_index=True)
display(comparison_results)

## 11. Sauvegarder les résultats de validation et le modèle LightGBM

Les fichiers seront enregistrés dans `data/processed/comparison_results`. Le modèle et les métriques sauvegardés facilitent la reproductibilité et la future comparaison avec les autres algorithmes.

In [ ]:
COMPARISON_DIR = PROCESSED_DIR / "comparison_results"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

lgbm_model_path = COMPARISON_DIR / "lightgbm_fraud_model_v1.txt"
lgbm_model.booster_.save_model(str(lgbm_model_path))

comparison_results.to_csv(COMPARISON_DIR / "validation_model_comparison.csv", index=False)
threshold_results_lgbm.to_csv(COMPARISON_DIR / "lightgbm_validation_thresholds.csv", index=False)
np.save(COMPARISON_DIR / "lightgbm_validation_scores.npy", y_validation_proba_lgbm)

metadata = {
    "model": "LightGBM",
    "purpose": "Première configuration comparative, non optimisée par recherche exhaustive",
    "train_period": "2010-2017",
    "validation_period": "2018",
    "test_period": "2019 — non chargé/utilisé dans cette étape",
    "n_features": int(X_train.shape[1]),
    "n_estimators": int(lgbm_model.n_estimators_),
    "scale_pos_weight": float(scale_pos_weight),
    "threshold_selected_on_validation_by_f1": best_threshold_lgbm,
    "validation_pr_auc": float(lgbm_validation_pr_auc),
    "validation_roc_auc": float(lgbm_validation_roc_auc),
    "training_minutes": float(training_seconds / 60),
    "parameters": lgbm_model.get_params(),
}
with open(COMPARISON_DIR / "lightgbm_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)

print("Fichiers sauvegardés dans :", COMPARISON_DIR)
for path in sorted(COMPARISON_DIR.iterdir()):
    print(f" - {path.name} ({path.stat().st_size / (1024**2):.2f} MB)")

## 9. Entraîner la régression logistique

### Objectif

La régression logistique constitue un modèle de référence (*baseline*) pour la détection des transactions frauduleuses. Elle estime la probabilité qu'une transaction appartienne à la classe fraude.

### Préparation et paramètres

- **Normalisation :** `StandardScaler(with_mean=False)` permet de mettre les variables à l'échelle tout en conservant la représentation sparse des données.
- **Gestion du déséquilibre :** `class_weight` donne davantage de poids aux fraudes, très minoritaires dans le jeu d'entraînement.
- **Optimisation :** le solveur `saga` prend en charge les matrices sparse.
- **Évaluation :** le modèle sera évalué sur la validation 2018 avec la PR-AUC, la ROC-AUC et les métriques au seuil de 0,30.

Le jeu de test 2019 reste réservé à l'évaluation finale.

In [ ]:

import time
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

# 1. Normalisation adaptée aux matrices sparse
scaler_lr = StandardScaler(with_mean=False)

X_train_lr = scaler_lr.fit_transform(X_train)
X_validation_lr = scaler_lr.transform(X_validation)

# 2. Définition du modèle
lr_model = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=1.0,
    class_weight={0: 1, 1: scale_pos_weight},
    max_iter=100,
    tol=1e-3,
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

# 3. Entraînement
start = time.time()

lr_model.fit(X_train_lr, y_train)

training_time_lr = (time.time() - start) / 60

# 4. Prédictions sur la validation 2018
y_validation_proba_lr = lr_model.predict_proba(
    X_validation_lr
)[:, 1]

# 5. Métriques globales
pr_auc_lr = average_precision_score(
    y_validation, y_validation_proba_lr
)
roc_auc_lr = roc_auc_score(
    y_validation, y_validation_proba_lr
)

# 6. Métriques au seuil de 0,30
threshold_lr = 0.30
y_validation_pred_lr = (
    y_validation_proba_lr >= threshold_lr
).astype(int)

precision_lr = precision_score(
    y_validation, y_validation_pred_lr, zero_division=0
)
recall_lr = recall_score(
    y_validation, y_validation_pred_lr, zero_division=0
)
f1_lr = f1_score(
    y_validation, y_validation_pred_lr, zero_division=0
)

# 7. Résultats
print("=== Validation 2018 — Régression logistique ===")
print(f"PR-AUC : {pr_auc_lr:.6f}")
print(f"ROC-AUC : {roc_auc_lr:.6f}")
print(f"Précision au seuil 0,30 : {precision_lr:.4%}")
print(f"Rappel au seuil 0,30 : {recall_lr:.4%}")
print(f"F1 au seuil 0,30 : {f1_lr:.6f}")
print(
    "Transactions signalées :",
    int(y_validation_pred_lr.sum())
)
print(f"Durée d'entraînement : {training_time_lr:.2f} minutes")
print("Itérations effectuées :", lr_model.n_iter_[0])


## 10. Optimiser le seuil de décision — Régression logistique

La régression logistique produit un score de fraude. Le seuil de 0,30 utilisé initialement génère un nombre excessif d'alertes.

Nous allons comparer plusieurs seuils sur la validation 2018 afin d'identifier celui qui offre le meilleur compromis entre précision, rappel et F1. Cette étape ne réentraîne pas le modèle et permet de déterminer si ses scores peuvent être exploités plus efficacement.

In [ ]:

import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
)

thresholds = [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]

results_lr_threshold = []

for threshold in thresholds:
    predictions = (
        y_validation_proba_lr >= threshold
    ).astype(int)

    results_lr_threshold.append({
        "Seuil": threshold,
        "Précision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Rappel": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "Transactions signalées": int(predictions.sum()),
    })

df_lr_thresholds = pd.DataFrame(results_lr_threshold)

display(
    df_lr_thresholds.style.format({
        "Seuil": "{:.2f}",
        "Précision": "{:.2%}",
        "Rappel": "{:.2%}",
        "F1": "{:.5f}",
        "Transactions signalées": "{:,.0f}",
    })
)

best_lr_threshold = df_lr_thresholds.loc[
    df_lr_thresholds["F1"].idxmax()
]

print("\n=== Meilleur seuil parmi ceux testés ===")
print(best_lr_threshold)



## 11. Entraîner le modèle Random Forest

La Random Forest combine plusieurs arbres de décision pour détecter les transactions frauduleuses. Nous utilisons `class_weight="balanced_subsample"` pour tenir compte du fort déséquilibre entre les transactions légitimes et frauduleuses.

Nous entraînons le modèle sur les données de 2010 à 2017, puis nous évaluons ses performances sur les données de validation de 2018. Nous conservons les données de test de 2019 pour l'évaluation finale.

In [ ]:
import time
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start = time.time()
rf_model.fit(X_train, y_train)
training_time_rf = (time.time() - start) / 60

y_validation_proba_rf = rf_model.predict_proba(X_validation)[:, 1]

pr_auc_rf = average_precision_score(
    y_validation, y_validation_proba_rf
)
roc_auc_rf = roc_auc_score(
    y_validation, y_validation_proba_rf
)

threshold_rf = 0.30
y_validation_pred_rf = (
    y_validation_proba_rf >= threshold_rf
).astype(int)

precision_rf = precision_score(
    y_validation, y_validation_pred_rf, zero_division=0
)
recall_rf = recall_score(
    y_validation, y_validation_pred_rf, zero_division=0
)
f1_rf = f1_score(
    y_validation, y_validation_pred_rf, zero_division=0
)

print("=== Validation 2018 — Random Forest ===")
print(f"PR-AUC : {pr_auc_rf:.6f}")
print(f"ROC-AUC : {roc_auc_rf:.6f}")
print(f"Précision au seuil 0,30 : {precision_rf:.4%}")
print(f"Rappel au seuil 0,30 : {recall_rf:.4%}")
print(f"F1 au seuil 0,30 : {f1_rf:.6f}")
print(f"Transactions signalées : {y_validation_pred_rf.sum():,}")
print(f"Durée d'entraînement : {training_time_rf:.2f} minutes")

## 12. Comparaison globale des modèles

Cette étape compare les performances de XGBoost, LightGBM, la régression logistique et Random Forest sur les données de validation de 2018. Les indicateurs retenus sont la PR-AUC, la ROC-AUC, la précision, le rappel et le score F1.

L'objectif est d'identifier le modèle le plus adapté à la détection des transactions frauduleuses dans un contexte de forte asymétrie des classes.

In [ ]:
import pandas as pd

# 1. Récupérer les métriques XGBoost déjà enregistrées
xgb_result = baseline_results.iloc[0]

# 2. Construire le tableau comparatif
comparaison_modeles = pd.DataFrame([
    {
        "Modèle": "XGBoost",
        "PR-AUC": xgb_result["validation_pr_auc"],
        "ROC-AUC": xgb_result["validation_roc_auc"],
        "Précision": xgb_result["precision_at_0_30"],
        "Rappel": xgb_result["recall_at_0_30"],
        "F1": xgb_result["f1_at_0_30"],
    },
    {
        "Modèle": "LightGBM",
        "PR-AUC": lgbm_validation_pr_auc,
        "ROC-AUC": lgbm_validation_roc_auc,
        "Précision": precision_lgbm,
        "Rappel": recall_lgbm,
        "F1": f1_lgbm,
    },
    {
        "Modèle": "Régression logistique",
        "PR-AUC": pr_auc_lr,
        "ROC-AUC": roc_auc_lr,
        "Précision": precision_lr,
        "Rappel": recall_lr,
        "F1": f1_lr,
    },
    {
        "Modèle": "Random Forest",
        "PR-AUC": pr_auc_rf,
        "ROC-AUC": roc_auc_rf,
        "Précision": precision_rf,
        "Rappel": recall_rf,
        "F1": f1_rf,
    },
])

display(
    comparaison_modeles.style.format(
        {
            "PR-AUC": "{:.6f}",
            "ROC-AUC": "{:.6f}",
            "Précision": "{:.2%}",
            "Rappel": "{:.2%}",
            "F1": "{:.5f}",
        },
        na_rep="—",
    )
)